# NB06 — Multi-Backbone Benchmark (leakage-controlled)

Five transformer backbones × four task variants, vanilla fine-tuning, matched protocol. Runs on **RunPod (GPU)**. Resumable: re-running skips completed (backbone, task) pairs.

Leakage is re-checked per task at load time (hard fail on overlap).

In [1]:
# Ensure dependencies (fast no-op if already installed on the pod)
import importlib, subprocess, sys
def ensure(pip_name, import_name=None):
    try:
        importlib.import_module(import_name or pip_name)
    except ImportError:
        print(f"installing {pip_name} ...")
        subprocess.run([sys.executable,"-m","pip","install","-q",pip_name], check=False)
for pkg,imp in [("transformers","transformers"),("datasets","datasets"),
                ("accelerate","accelerate"),("scikit-learn","sklearn"),
                ("pandas","pandas"),("numpy","numpy")]:
    ensure(pkg,imp)
print("deps ok")


deps ok


In [2]:
import os, json, time, random, shutil, warnings, unicodedata, re, gc
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             confusion_matrix, classification_report)
warnings.filterwarnings("ignore")
import transformers
print("torch:", torch.__version__, "| transformers:", transformers.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


torch: 2.8.0+cu128 | transformers: 4.40.0
device: cuda
GPU: NVIDIA RTX A5000 25.4 GB


In [3]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("seed set:", SEED)


seed set: 42


In [4]:
def find_repo_root():
    cands = [Path("/workspace/Sarcasm_detection"), Path.cwd()] + list(Path.cwd().resolve().parents)
    for c in cands:
        if (c / "01_data").exists():
            return c.resolve()
    raise RuntimeError("Could not locate repo root (no 01_data dir found). Run from inside the repo.")

ROOT       = find_repo_root()
SPLITS     = ROOT / "01_data" / "interim" / "splits"
CKPT       = ROOT / "03_checkpoints"
TABLES     = ROOT / "04_outputs" / "tables"
PRED       = ROOT / "04_outputs" / "predictions"
LOGS       = ROOT / "04_outputs" / "run_logs"
REPORTS    = ROOT / "04_outputs" / "reports"
for p in [CKPT, TABLES, PRED, LOGS, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)
print("ROOT  :", ROOT)
print("SPLITS:", SPLITS, "| exists:", SPLITS.exists())
assert SPLITS.exists(), f"Splits folder missing: {SPLITS}. Run NB01 first."


ROOT  : /workspace/Sarcasm_detection
SPLITS: /workspace/Sarcasm_detection/01_data/interim/splits | exists: True


In [5]:
# ---- normalized key + leakage guard (defense-in-depth; NB01 already enforces this) ----
_ZW = dict.fromkeys(map(ord, ["\u200b","\u200c","\u200d","\ufeff"]), None)
def norm_key(s):
    if not isinstance(s, str):
        s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFC", s).translate(_ZW)
    s = re.sub(r"\s+", " ", s).strip()
    return s.casefold()

def assert_no_internal_leakage(tr, va, te, text_col="text"):
    s_tr = {norm_key(x) for x in tr[text_col]}
    s_va = {norm_key(x) for x in va[text_col]}
    s_te = {norm_key(x) for x in te[text_col]}
    n1, n2, n3 = len(s_tr & s_va), len(s_tr & s_te), len(s_va & s_te)
    assert n1 == 0, f"LEAK train/val overlap = {n1}"
    assert n2 == 0, f"LEAK train/test overlap = {n2}"
    assert n3 == 0, f"LEAK val/test overlap = {n3}"
    return {"train_val": n1, "train_test": n2, "val_test": n3}

def softmax_np(x):
    x = np.asarray(x, dtype=np.float64)
    x = x - x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

def eval_from_logits(labels, logits, num_labels):
    labels = np.asarray(labels)
    preds = np.asarray(logits).argmax(-1)
    m = {"accuracy": float(accuracy_score(labels, preds)),
         "macro_f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
         "weighted_f1": float(f1_score(labels, preds, average="weighted", zero_division=0))}
    if num_labels == 2:
        m["binary_f1"]  = float(f1_score(labels, preds, average="binary", pos_label=1, zero_division=0))
        m["precision"]  = float(precision_score(labels, preds, average="binary", pos_label=1, zero_division=0))
        m["recall"]     = float(recall_score(labels, preds, average="binary", pos_label=1, zero_division=0))
    return m, preds

def make_compute_metrics(num_labels):
    def _cm(eval_pred):
        logits, labels = eval_pred
        preds = np.asarray(logits).argmax(-1)
        return {"accuracy": accuracy_score(labels, preds),
                "macro_f1": f1_score(labels, preds, average="macro", zero_division=0)}
    return _cm

def save_predictions(path, texts, labels, logits, num_labels, extra=None):
    logits = np.asarray(logits); probs = softmax_np(logits); preds = logits.argmax(-1)
    d = {"text": [str(t) for t in texts], "gold_label": np.asarray(labels),
         "pred_label": preds, "correct": (np.asarray(labels) == preds).astype(int)}
    for k in range(num_labels): d[f"logit_{k}"] = logits[:, k]
    for k in range(num_labels): d[f"prob_{k}"]  = probs[:, k]
    df = pd.DataFrame(d)
    if extra:
        for k, v in extra.items(): df[k] = v
    df.to_csv(path, index=False)

class TorchTextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.enc = tokenizer(list(texts), truncation=True, padding="max_length",
                             max_length=max_length, return_tensors="pt")
        self.labels = torch.tensor(list(labels), dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

def build_args(output_dir, epochs, batch, lr, wd, warmup, fp16, metric="macro_f1"):
    common = dict(output_dir=str(output_dir), num_train_epochs=epochs,
                  per_device_train_batch_size=batch, per_device_eval_batch_size=batch*2,
                  learning_rate=lr, weight_decay=wd, warmup_ratio=warmup,
                  save_strategy="epoch", logging_strategy="epoch", save_total_limit=1,
                  load_best_model_at_end=True, metric_for_best_model=metric,
                  greater_is_better=True, report_to="none", seed=SEED, data_seed=SEED,
                  fp16=fp16, bf16=False)
    try:
        return TrainingArguments(evaluation_strategy="epoch", **common)
    except TypeError:
        return TrainingArguments(eval_strategy="epoch", **common)


In [6]:
TASKS = {
    "ben_sarc_binary":     {"label_col": "label_binary",  "num_labels": 2},
    "banglasarc_binary":   {"label_col": "label_binary",  "num_labels": 2},
    "banglasarc3_binary":  {"label_col": "label_binary",  "num_labels": 2},
    "banglasarc3_ternary": {"label_col": "label_ternary", "num_labels": 3},
}

def load_task(task):
    cfg = TASKS[task]; lc = cfg["label_col"]
    def rd(split):
        df = pd.read_csv(SPLITS / f"{task}_{split}.csv")
        assert "text" in df.columns,  f"{task}_{split}: missing 'text'"
        assert lc in df.columns,      f"{task}_{split}: missing '{lc}'"
        df = df[["text", lc]].dropna(subset=["text"]).copy()
        df["text"] = df["text"].astype(str)
        df[lc] = df[lc].astype(int)
        return df
    tr, va, te = rd("train"), rd("val"), rd("test")
    leak = assert_no_internal_leakage(tr, va, te)   # hard fail on leakage
    return tr, va, te, lc, cfg["num_labels"], leak


In [7]:
# ---- CONFIG ----
BACKBONES = {
    "banglabert":     "csebuetnlp/banglabert",
    "muril":          "google/muril-base-cased",
    "xlm-roberta":    "xlm-roberta-base",
    "mbert":          "bert-base-multilingual-cased",
    "sagorsarker-bb": "sagorsarker/bangla-bert-base",
}
MAX_LENGTH = 128
BATCH_SIZE = 32
EPOCHS     = 4
LR         = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
PATIENCE   = 2
USE_FP16   = torch.cuda.is_available()
print(f"runs = {len(BACKBONES)} backbones x {len(TASKS)} tasks = {len(BACKBONES)*len(TASKS)}")


runs = 5 backbones x 4 tasks = 20


In [8]:
def train_eval(model_tag, model_name, task):
    tr, va, te, lc, num_labels, leak = load_task(task)
    tok = AutoTokenizer.from_pretrained(model_name)
    train_ds = TorchTextDataset(tr["text"], tr[lc], tok, MAX_LENGTH)
    val_ds   = TorchTextDataset(va["text"], va[lc], tok, MAX_LENGTH)
    test_ds  = TorchTextDataset(te["text"], te[lc], tok, MAX_LENGTH)

    out_dir = CKPT / f"06_tmp_{model_tag}_{task}"
    if out_dir.exists(): shutil.rmtree(out_dir, ignore_errors=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_labels, ignore_mismatched_sizes=True)
    args  = build_args(out_dir, EPOCHS, BATCH_SIZE, LR, WEIGHT_DECAY, WARMUP_RATIO, USE_FP16)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      compute_metrics=make_compute_metrics(num_labels),
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    t0 = time.time(); trainer.train(); secs = round(time.time()-t0,1)

    val_logits  = trainer.predict(val_ds).predictions
    test_logits = trainer.predict(test_ds).predictions
    val_m,  _          = eval_from_logits(va[lc].values, val_logits,  num_labels)
    test_m, test_preds = eval_from_logits(te[lc].values, test_logits, num_labels)

    save_predictions(PRED / f"06_{model_tag}_{task}_test_predictions.csv",
                     te["text"].values, te[lc].values, test_logits, num_labels,
                     extra={"backbone": model_tag, "task": task})
    cm = confusion_matrix(te[lc].values, test_preds).tolist()

    row = {"backbone": model_tag, "model_name": model_name, "task": task, "num_labels": num_labels,
           "n_train": len(tr), "n_test": len(te), "time_seconds": secs,
           "confusion_matrix": json.dumps(cm),
           "leak_train_test": leak["train_test"], "leak_val_test": leak["val_test"]}
    for k,v in val_m.items():  row[f"val_{k}"]  = v
    for k,v in test_m.items(): row[f"test_{k}"] = v

    del model, trainer; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    shutil.rmtree(out_dir, ignore_errors=True)
    return row


In [9]:
SUMMARY = TABLES / "06_multi_backbone_summary.csv"
rows, done = [], set()
if SUMMARY.exists():
    prev = pd.read_csv(SUMMARY); rows = prev.to_dict("records")
    done = set(zip(prev["backbone"], prev["task"])); print("resuming, done pairs:", len(done))

n = 0; total = len(BACKBONES)*len(TASKS)
for mtag, mname in BACKBONES.items():
    for task in TASKS:
        n += 1
        if (mtag, task) in done: 
            print(f"[{n}/{total}] skip {mtag} x {task}"); continue
        print(f"\n[{n}/{total}] {mtag} x {task}  ({mname})")
        try:
            rows.append(train_eval(mtag, mname, task))
            pd.DataFrame(rows).to_csv(SUMMARY, index=False)
            print("  saved.")
        except Exception as e:
            print("  FAILED:", repr(e))
            if rows: pd.DataFrame(rows).to_csv(SUMMARY, index=False)
            raise
print("\nALL RUNS DONE")



[1/20] banglabert x ben_sarc_binary  (csebuetnlp/banglabert)


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.536900,0.435886,0.795472,0.795185
2,0.379800,0.464092,0.793911,0.793276
3,0.258800,0.550088,0.787666,0.786916


  saved.

[2/20] banglabert x banglasarc_binary  (csebuetnlp/banglabert)


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.614300,0.557710,0.711757,0.709142
2,0.479600,0.566280,0.713021,0.710624
3,0.349900,0.630293,0.718078,0.718050
4,0.232700,0.731413,0.697851,0.697694


  saved.

[20/20] sagorsarker-bb x banglasarc3_ternary  (sagorsarker/bangla-bert-base)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sagorsarker/bangla-bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.966000,0.900261,0.583543,0.558765
2,0.785600,0.870981,0.600336,0.591654
3,0.618100,0.946075,0.600336,0.600498
4,0.469000,0.998073,0.605374,0.604966


  saved.

ALL RUNS DONE


In [10]:
res = pd.read_csv(SUMMARY)
print(res[["backbone","task","test_accuracy","test_macro_f1","time_seconds"]].to_string(index=False))

pivot = res.pivot_table(index="backbone", columns="task", values="test_macro_f1", aggfunc="first")
pivot["mean"] = pivot.mean(axis=1)
pivot = pivot.sort_values("mean", ascending=False)
pivot.to_csv(TABLES / "06_macro_f1_pivot.csv")
print("\nMACRO-F1 by backbone x task:\n")
print(pivot.round(4).to_string())
print("\nbest backbone (mean macro-F1):", pivot["mean"].idxmax(), round(pivot["mean"].max(),4))

cm_dict = {f"{r.backbone}_{r.task}": json.loads(r.confusion_matrix) for r in res.itertuples()}
json.dump(cm_dict, open(TABLES / "06_confusion_matrices.json","w"), indent=2)
print("saved pivot + confusion matrices")


      backbone                task  test_accuracy  test_macro_f1  time_seconds
    banglabert     ben_sarc_binary       0.798283       0.798066         183.5
    banglabert   banglasarc_binary       0.976293       0.974240          57.8
    banglabert  banglasarc3_binary       0.737042       0.736526          86.9
    banglabert banglasarc3_ternary       0.640101       0.635114         123.2
         muril     ben_sarc_binary       0.797113       0.797102         311.7
         muril   banglasarc_binary       0.969828       0.967171          83.9
         muril  banglasarc3_binary       0.747155       0.746815         122.4
         muril banglasarc3_ternary       0.624161       0.618725         169.7
   xlm-roberta     ben_sarc_binary       0.780726       0.780716         335.7
   xlm-roberta   banglasarc_binary       0.937500       0.932776          92.6
   xlm-roberta  banglasarc3_binary       0.729456       0.729141         130.0
   xlm-roberta banglasarc3_ternary       0.640101   